<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Gemini API and VertexAI</h1>
<h1>End-to-End Generative AI Solutions</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
from collections import Counter
from pprint import pprint
import os
import json
import hashlib
import time
import re
from typing import Optional, List, Dict, Any
from dataclasses import dataclass, field
from enum import Enum
import logging

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import google.generativeai as genai
from google.generativeai.types import HarmCategory, HarmBlockThreshold
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig, Part

import watermark
%load_ext watermark
%matplotlib inline


/var/folders/lr/j1bs1q851k15cj5y777nxwph0000gn/T/ipykernel_90773/1467124714.py:18: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


We start by printing out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.10.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 0f22cb8c30fea71d49b61d9441111870a39cba67

json      : 2.0.9
logging   : 0.5.1.2
matplotlib: 3.10.8
numpy     : 2.4.2
pandas    : 3.0.1
re        : 2.2.1
vertexai  : 1.71.1
watermark : 2.6.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# 1. Designing and Architecting Generative AI Systems

In this segment we cover how to design and build complete, production-ready Generative AI applications using the Gemini API and Vertex AI. We explore three foundational architecture patterns and then implement them end-to-end.

## 1a. Architecture Patterns Overview

There are three dominant architecture patterns for Generative AI systems:

| Pattern | Description | Best For |
|---|---|---|
| **RAG** | Retrieve relevant external knowledge, then generate | Knowledge-intensive QA, document search |
| **Agent** | LLM decides which tools to call and when | Multi-step tasks, tool orchestration |
| **Pipeline** | Chain of LLM calls, each refining the output | Complex transformations, multi-stage reasoning |

We will implement all three patterns and show how they can be combined.

In [ ]:
# Visualize the three architecture patterns
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Generative AI Architecture Patterns', fontsize=14, fontweight='bold')

arch_colors = {
    'user':      colors[0],
    'llm':       colors[2],
    'retriever': colors[3],
    'vectordb':  colors[1],
    'tools':     colors[5],
    'output':    colors[6],
    'step':      colors[8],
}

def draw_box(ax, x, y, w, h, label, color, fontsize=9):
    rect = mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle='round,pad=0.05',
        facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9
    )
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=fontsize,
            color='white', fontweight='bold', wrap=True)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=colors[7], lw=1.5))

# --- RAG Pattern ---
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('RAG Pattern', fontsize=12, fontweight='bold', pad=10)
draw_box(ax, 5, 9,   4, 1,   'User Query',        arch_colors['user'])
draw_box(ax, 5, 7,   4, 1,   'Retriever',         arch_colors['retriever'])
draw_box(ax, 5, 5,   4, 1,   'Vector DB',         arch_colors['vectordb'])
draw_box(ax, 5, 3,   4, 1,   'LLM (Gemini)',      arch_colors['llm'])
draw_box(ax, 5, 1,   4, 1,   'Answer',            arch_colors['output'])
draw_arrow(ax, 5, 8.5, 5, 7.5)
draw_arrow(ax, 5, 6.5, 5, 5.5)
draw_arrow(ax, 5, 4.5, 5, 3.5)
draw_arrow(ax, 5, 2.5, 5, 1.5)
ax.annotate('', xy=(7.5, 3.0), xytext=(7.5, 5.0),
            arrowprops=dict(arrowstyle='->', color=arch_colors['retriever'], lw=1.5, linestyle='dashed'))
ax.text(8.2, 4.0, 'Context', fontsize=9, color=arch_colors['retriever'], rotation=90)

# --- Agent Pattern ---
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('Agent Pattern', fontsize=12, fontweight='bold', pad=10)
draw_box(ax, 5, 9,   4, 1,   'User Goal',         arch_colors['user'])
draw_box(ax, 5, 7,   4, 1,   'LLM (Gemini)',      arch_colors['llm'])
draw_box(ax, 2, 4.5, 3, 1,   'Calculator',        arch_colors['tools'])
draw_box(ax, 5, 4.5, 3, 1,   'Search',            arch_colors['tools'])
draw_box(ax, 8, 4.5, 3, 1,   'Database',          arch_colors['tools'])
draw_box(ax, 5, 2,   4, 1,   'Tool Results',      arch_colors['retriever'])
draw_box(ax, 5, 0.5, 4, 1,   'Final Answer',      arch_colors['output'])
draw_arrow(ax, 5, 8.5, 5, 7.5)
draw_arrow(ax, 3.5, 6.5, 2, 5.0)
draw_arrow(ax, 5.0, 6.5, 5, 5.0)
draw_arrow(ax, 6.5, 6.5, 8, 5.0)
draw_arrow(ax, 2, 4.0, 3.5, 2.5)
draw_arrow(ax, 5, 4.0, 5, 2.5)
draw_arrow(ax, 8, 4.0, 6.5, 2.5)
draw_arrow(ax, 5, 1.5, 5, 1.0)

# --- Pipeline Pattern ---
ax = axes[2]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('Pipeline Pattern', fontsize=12, fontweight='bold', pad=10)
steps = [
    (5, 9.0, 'Input'),
    (5, 7.2, 'Step 1: Extract'),
    (5, 5.4, 'Step 2: Transform'),
    (5, 3.6, 'Step 3: Validate'),
    (5, 1.8, 'Output'),
]
step_colors = [arch_colors['user'], arch_colors['step'], arch_colors['step'], arch_colors['step'], arch_colors['output']]
for (x, y, label), c in zip(steps, step_colors):
    draw_box(ax, x, y, 4, 1, label, c)
for i in range(len(steps) - 1):
    draw_arrow(ax, steps[i][0], steps[i][1] - 0.5, steps[i+1][0], steps[i+1][1] + 0.5)

plt.tight_layout()
plt.savefig('architecture_patterns.png', dpi=120, bbox_inches='tight')
plt.show()

## 1b. RAG (Retrieval-Augmented Generation) Implementation

RAG solves the "knowledge cutoff" and "hallucination" problems by grounding the LLM's responses in retrieved documents. The pipeline is:

1. **Index**: Embed all documents into a vector store
2. **Retrieve**: Given a query, find the most similar document chunks
3. **Generate**: Feed the retrieved context + query to the LLM

In [ ]:
# Configure Gemini API
# In production, use Secret Manager or environment variables
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
PROJECT_ID     = os.environ.get('GOOGLE_CLOUD_PROJECT', 'your-project-id')
LOCATION       = os.environ.get('GOOGLE_CLOUD_LOCATION', 'us-central1')

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print('Configured with API key')
else:
    # Vertex AI uses Application Default Credentials
    vertexai.init(project=PROJECT_ID, location=LOCATION)
    print(f'Configured with Vertex AI (project={PROJECT_ID}, location={LOCATION})')

In [6]:
class SimpleRAG:
    """
    A minimal Retrieval-Augmented Generation (RAG) system backed by an
    in-memory numpy vector store and Gemini embeddings.
    """

    def __init__(self, model_name: str = 'gemini-2.5-flash',
                 embedding_model: str = 'models/text-embedding-004'):
        self.model_name      = model_name
        self.embedding_model = embedding_model
        self.documents: List[str]        = []
        self.embeddings: Optional[np.ndarray] = None
        self.model = genai.GenerativeModel(model_name)

    # ------------------------------------------------------------------
    # Indexing
    # ------------------------------------------------------------------
    def _embed(self, texts: List[str]) -> np.ndarray:
        """Embed a list of texts using the Gemini embedding model."""
        result = genai.embed_content(
            model=self.embedding_model,
            content=texts,
            task_type='retrieval_document'
        )
        return np.array(result['embedding'])

    def add_documents(self, documents: List[str]) -> None:
        """Embed and store a list of document chunks."""
        print(f'Embedding {len(documents)} document(s)...')
        new_embeddings = self._embed(documents)
        self.documents.extend(documents)
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
        print(f'Vector store now contains {len(self.documents)} chunk(s).')

    # ------------------------------------------------------------------
    # Retrieval
    # ------------------------------------------------------------------
    @staticmethod
    def _cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
        """Compute cosine similarity between vector a and matrix b."""
        a_norm = a / (np.linalg.norm(a) + 1e-10)
        b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-10)
        return b_norm @ a_norm

    def retrieve(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        """Retrieve the top_k most relevant document chunks."""
        if self.embeddings is None or len(self.documents) == 0:
            raise ValueError('No documents indexed. Call add_documents() first.')

        query_emb = self._embed([query])
        scores    = self._cosine_similarity(query_emb[0], self.embeddings)
        top_idx   = np.argsort(scores)[::-1][:top_k]

        return [
            {'document': self.documents[i], 'score': float(scores[i])}
            for i in top_idx
        ]

    # ------------------------------------------------------------------
    # Generation
    # ------------------------------------------------------------------
    def generate(self, query: str, top_k: int = 3) -> Dict[str, Any]:
        """Full RAG pipeline: retrieve then generate."""
        retrieved = self.retrieve(query, top_k=top_k)
        context   = '\n\n'.join(
            f'[Source {i+1} (score={r["score"]:.3f})]:\n{r["document"]}'
            for i, r in enumerate(retrieved)
        )

        prompt = (
            f'You are a helpful assistant. Use ONLY the context below to answer '
            f'the question. If the answer is not in the context, say so.\n\n'
            f'CONTEXT:\n{context}\n\n'
            f'QUESTION: {query}\n\n'
            f'ANSWER:'
        )

        response = self.model.generate_content(prompt)
        return {
            'query':     query,
            'retrieved': retrieved,
            'answer':    response.text
        }

In [7]:
# Sample knowledge base – product documentation snippets
PRODUCT_DOCS = [
    """Gemini 2.0 Flash is Google's latest multimodal AI model. It supports text, 
    images, audio, and video inputs. The context window is 1 million tokens.""",

    """Vertex AI provides enterprise-grade infrastructure for deploying Gemini models.
    It includes built-in security, compliance, and monitoring tools. It supports
    IAM-based access control and VPC Service Controls.""",

    """The Gemini API supports function calling, allowing the model to invoke
    user-defined functions and use their results in responses. Functions are
    defined using JSON schemas.""",

    """Pricing for Gemini 2.0 Flash is $0.075 per 1M input tokens and $0.30 per
    1M output tokens. There are also separate rates for image, audio and video inputs.""",

    """Gemini Embedding models convert text into dense vector representations.
    The text-embedding-004 model produces 768-dimensional embeddings and supports
    task types: retrieval_document, retrieval_query, semantic_similarity, and classification.""",

    """Safety settings in the Gemini API allow developers to configure thresholds
    for harmful content categories: HARASSMENT, HATE_SPEECH, SEXUALLY_EXPLICIT,
    and DANGEROUS_CONTENT. Each can be set to BLOCK_NONE, BLOCK_LOW_AND_ABOVE, etc.""",
]

# Instantiate and populate the RAG system
rag = SimpleRAG()
rag.add_documents(PRODUCT_DOCS)

Embedding 6 document(s)...


NotFound: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.

In [ ]:
# Test the RAG pipeline
test_queries = [
    'What is the context window size of Gemini 2.0 Flash?',
    'How much does the Gemini API cost?',
    'How do I control safety in the Gemini API?',
]

for query in test_queries:
    print('=' * 70)
    result = rag.generate(query)
    print(f'QUERY: {result["query"]}')
    print(f'\nTOP RETRIEVED CHUNKS:')
    for r in result['retrieved']:
        print(f'  [score={r["score"]:.3f}] {r["document"][:80].strip()}...')
    print(f'\nANSWER: {result["answer"]}')
    print()

## 1c. Function Calling / Tool Use with Gemini

Gemini's function calling lets the model decide when and how to invoke external tools. The agent loop:

1. User sends a message
2. Gemini returns a `function_call` part instead of text
3. We execute the function and send back a `function_response`
4. Gemini generates the final answer incorporating the tool result
5. Repeat until Gemini returns plain text (task complete)

In [ ]:
# --- Tool Definitions ---

def get_weather(city: str) -> Dict[str, Any]:
    """Mock weather API."""
    mock_data = {
        'New York':    {'temp_c': 22, 'condition': 'Partly cloudy', 'humidity': 65},
        'London':      {'temp_c': 15, 'condition': 'Rainy',         'humidity': 80},
        'Tokyo':       {'temp_c': 28, 'condition': 'Sunny',         'humidity': 70},
        'San Francisco': {'temp_c': 18, 'condition': 'Foggy',       'humidity': 75},
    }
    data = mock_data.get(city, {'temp_c': 20, 'condition': 'Unknown', 'humidity': 60})
    return {'city': city, **data}


def calculator(expression: str) -> Dict[str, Any]:
    """Safe arithmetic evaluator."""
    allowed = set('0123456789+-*/().^ ')
    if not all(c in allowed for c in expression):
        return {'error': 'Invalid characters in expression'}
    try:
        result = eval(expression, {'__builtins__': {}})  # noqa: S307
        return {'expression': expression, 'result': result}
    except Exception as e:
        return {'error': str(e)}


def query_database(table: str, filter_field: str = '', filter_value: str = '') -> Dict[str, Any]:
    """Mock database query."""
    db = {
        'products': [
            {'id': 1, 'name': 'Laptop Pro',  'price': 1299, 'category': 'Electronics'},
            {'id': 2, 'name': 'Wireless Earbuds', 'price': 199, 'category': 'Electronics'},
            {'id': 3, 'name': 'Coffee Maker',  'price': 89,  'category': 'Appliances'},
        ],
        'customers': [
            {'id': 1, 'name': 'Alice',  'tier': 'Gold',   'orders': 12},
            {'id': 2, 'name': 'Bob',    'tier': 'Silver', 'orders': 5},
        ]
    }
    rows = db.get(table, [])
    if filter_field and filter_value:
        rows = [r for r in rows if str(r.get(filter_field, '')).lower() == filter_value.lower()]
    return {'table': table, 'rows': rows, 'count': len(rows)}


# Registry mapping tool names to Python callables
TOOL_REGISTRY: Dict[str, Any] = {
    'get_weather':    get_weather,
    'calculator':     calculator,
    'query_database': query_database,
}

print('Tools registered:', list(TOOL_REGISTRY.keys()))

In [ ]:
# Define the tool schemas for Gemini function calling
tools = [
    genai.protos.Tool(
        function_declarations=[
            genai.protos.FunctionDeclaration(
                name='get_weather',
                description='Get current weather information for a given city.',
                parameters=genai.protos.Schema(
                    type=genai.protos.Type.OBJECT,
                    properties={
                        'city': genai.protos.Schema(
                            type=genai.protos.Type.STRING,
                            description='The city name, e.g. New York'
                        )
                    },
                    required=['city']
                )
            ),
            genai.protos.FunctionDeclaration(
                name='calculator',
                description='Evaluate a mathematical expression and return the result.',
                parameters=genai.protos.Schema(
                    type=genai.protos.Type.OBJECT,
                    properties={
                        'expression': genai.protos.Schema(
                            type=genai.protos.Type.STRING,
                            description='A valid arithmetic expression, e.g. (3 + 4) * 2'
                        )
                    },
                    required=['expression']
                )
            ),
            genai.protos.FunctionDeclaration(
                name='query_database',
                description='Query a mock database table, optionally filtering by a field value.',
                parameters=genai.protos.Schema(
                    type=genai.protos.Type.OBJECT,
                    properties={
                        'table': genai.protos.Schema(
                            type=genai.protos.Type.STRING,
                            description='Table name: products or customers'
                        ),
                        'filter_field': genai.protos.Schema(
                            type=genai.protos.Type.STRING,
                            description='Field to filter on (optional)'
                        ),
                        'filter_value': genai.protos.Schema(
                            type=genai.protos.Type.STRING,
                            description='Value to filter for (optional)'
                        ),
                    },
                    required=['table']
                )
            ),
        ]
    )
]

agent_model = genai.GenerativeModel('gemini-2.5-flash', tools=tools)
print('Agent model configured with tools.')

In [ ]:
def run_agent(user_message: str, max_iterations: int = 5) -> str:
    """
    Simple agent loop: send message, handle function calls, return final answer.
    """
    chat    = agent_model.start_chat()
    message = user_message
    print(f'USER: {user_message}\n')

    for iteration in range(max_iterations):
        response = chat.send_message(message)
        part     = response.candidates[0].content.parts[0]

        # If the model returned plain text, we are done
        if hasattr(part, 'text') and part.text:
            print(f'AGENT: {part.text}')
            return part.text

        # Otherwise handle function calls
        if hasattr(part, 'function_call') and part.function_call:
            fc       = part.function_call
            fn_name  = fc.name
            fn_args  = dict(fc.args)
            print(f'  [Tool call] {fn_name}({fn_args})')

            fn = TOOL_REGISTRY.get(fn_name)
            if fn is None:
                fn_result = {'error': f'Unknown tool: {fn_name}'}
            else:
                fn_result = fn(**fn_args)

            print(f'  [Tool result] {fn_result}')

            # Feed the function result back to the model
            message = genai.protos.Content(
                parts=[
                    genai.protos.Part(
                        function_response=genai.protos.FunctionResponse(
                            name=fn_name,
                            response={'result': fn_result}
                        )
                    )
                ],
                role='user'
            )

    return 'Agent reached maximum iterations without a final answer.'


# Test the agent
_ = run_agent('What is the weather in Tokyo, and what is 1234 * 5678?')

In [ ]:
# Another agent test – multi-step database query
_ = run_agent(
    'Show me all Electronics products in the database, '
    'and tell me the total price if someone buys all of them.'
)

## 1d. Pipeline Pattern: Chained LLM Calls

Some tasks are too complex for a single prompt. A pipeline breaks the work into discrete LLM-powered stages, each building on the previous output. This improves reliability and makes each step auditable.

In [ ]:
class LLMPipeline:
    """A simple chained LLM pipeline for multi-stage text processing."""

    def __init__(self, model_name: str = 'gemini-2.5-flash'):
        self.model  = genai.GenerativeModel(model_name)
        self.stages: List[Dict[str, Any]] = []

    def _call(self, prompt: str) -> str:
        return self.model.generate_content(prompt).text.strip()

    def run(self, raw_text: str) -> Dict[str, Any]:
        results = {'input': raw_text}

        # Stage 1: Extract key facts
        print('Stage 1: Extracting key facts...')
        facts = self._call(
            f'Extract the 3-5 most important facts from this text as a JSON list of strings.\n\n'
            f'TEXT:\n{raw_text}\n\nJSON list:'
        )
        results['facts'] = facts

        # Stage 2: Classify sentiment and topic
        print('Stage 2: Classifying...')
        classification = self._call(
            f'Given these facts: {facts}\n\n'
            f'Return a JSON object with keys: sentiment (positive/neutral/negative), '
            f'topic (one word), confidence (0.0-1.0).'
        )
        results['classification'] = classification

        # Stage 3: Generate executive summary
        print('Stage 3: Writing summary...')
        summary = self._call(
            f'Write a 2-sentence executive summary based on:\n'
            f'Facts: {facts}\n'
            f'Classification: {classification}\n'
            f'Keep it professional and concise.'
        )
        results['summary'] = summary

        print('Pipeline complete.')
        return results


sample_text = """
Our Q4 results exceeded expectations with revenue growing 34% year-over-year to $2.8B.
Customer acquisition costs dropped 18% due to improved targeting in our AI-powered
ad campaigns. However, supply chain disruptions impacted hardware margins by 3 points.
We are guiding for 28-32% revenue growth in Q1 of the following year.
"""

pipeline  = LLMPipeline()
pipeline_results = pipeline.run(sample_text)

print('\n--- Pipeline Results ---')
for stage, value in pipeline_results.items():
    print(f'\n[{stage.upper()}]')
    print(value)

## 1e. Evaluation Framework: LLM-as-Judge

Evaluating LLM outputs is hard. We use **LLM-as-judge**: a separate Gemini call scores each response on defined criteria. Key metrics:

- **Faithfulness**: Is the answer grounded in the retrieved context?
- **Relevance**: Does the answer address the question?
- **Coherence**: Is the answer well-structured and fluent?

In [ ]:
@dataclass
class EvalResult:
    faithfulness: float
    relevance:    float
    coherence:    float
    reasoning:    str

    @property
    def overall(self) -> float:
        return round((self.faithfulness + self.relevance + self.coherence) / 3, 3)


class LLMEvaluator:
    """Evaluates RAG outputs using Gemini as a judge."""

    EVAL_PROMPT = """You are an expert evaluator for AI-generated answers.

Evaluate the answer below on three criteria, each scored 0.0 to 1.0:
- faithfulness: Is every claim in the answer supported by the provided context?
- relevance:    Does the answer directly address the question?
- coherence:    Is the answer well-structured, clear, and fluent?

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
{answer}

Return ONLY valid JSON in this exact format (no markdown, no extra text):
{{"faithfulness": 0.0, "relevance": 0.0, "coherence": 0.0, "reasoning": "..."}}"""

    def __init__(self, model_name: str = 'gemini-2.5-flash'):
        self.model = genai.GenerativeModel(
            model_name,
            generation_config=GenerationConfig(temperature=0.0)
        )

    def evaluate(self, question: str, context: str, answer: str) -> EvalResult:
        prompt   = self.EVAL_PROMPT.format(
            context=context, question=question, answer=answer
        )
        response = self.model.generate_content(prompt)
        raw      = response.text.strip()

        # Strip markdown code fences if present
        raw = re.sub(r'^```json\s*|```\s*$', '', raw, flags=re.MULTILINE).strip()

        data = json.loads(raw)
        return EvalResult(
            faithfulness=float(data.get('faithfulness', 0)),
            relevance=float(data.get('relevance', 0)),
            coherence=float(data.get('coherence', 0)),
            reasoning=data.get('reasoning', '')
        )

    def batch_evaluate(self, samples: List[Dict[str, str]]) -> pd.DataFrame:
        records = []
        for sample in samples:
            result = self.evaluate(
                question=sample['question'],
                context=sample['context'],
                answer=sample['answer']
            )
            records.append({
                'question':     sample['question'],
                'faithfulness': result.faithfulness,
                'relevance':    result.relevance,
                'coherence':    result.coherence,
                'overall':      result.overall,
                'reasoning':    result.reasoning,
            })
        return pd.DataFrame(records)

In [ ]:
# Evaluation dataset: mix of good and bad answers
eval_samples = [
    {
        'question': 'What is the context window of Gemini 2.0 Flash?',
        'context':  'Gemini 2.0 Flash supports text, images, audio, and video. The context window is 1 million tokens.',
        'answer':   'Gemini 2.0 Flash has a 1 million token context window.',
    },
    {
        'question': 'What does Vertex AI provide?',
        'context':  'Vertex AI provides enterprise-grade infrastructure for deploying Gemini models, with built-in security and monitoring.',
        'answer':   'Vertex AI is a consumer product that allows anyone to use Gemini for free with unlimited tokens.',
    },
    {
        'question': 'How does Gemini function calling work?',
        'context':  'Function calling allows the model to invoke user-defined functions. Functions are defined using JSON schemas.',
        'answer':   'Function calling in Gemini allows the model to invoke user-defined functions defined via JSON schemas.',
    },
]

evaluator = LLMEvaluator()
eval_df   = evaluator.batch_evaluate(eval_samples)
print(eval_df[['question', 'faithfulness', 'relevance', 'coherence', 'overall']].to_string(index=False))

In [ ]:
# Visualize evaluation results
metrics   = ['faithfulness', 'relevance', 'coherence']
x         = np.arange(len(eval_df))
bar_width  = 0.25
colors_bar = [colors[0], colors[2], colors[3]]

fig, ax = plt.subplots(figsize=(11, 5))
for i, (metric, color) in enumerate(zip(metrics, colors_bar)):
    bars = ax.bar(x + i * bar_width, eval_df[metric], bar_width,
                  label=metric.capitalize(), color=color, alpha=0.85)
    ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)

ax.set_xlabel('Sample', fontsize=10)
ax.set_ylabel('Score (0–1)', fontsize=10)
ax.set_title('LLM-as-Judge Evaluation Results', fontsize=12)
ax.set_xticks(x + bar_width)
ax.set_xticklabels(
    [f'Sample {i+1}' for i in range(len(eval_df))], fontsize=10
)
ax.tick_params(axis='y', labelsize=10)
ax.set_ylim(0, 1.2)
ax.legend(fontsize=10)
ax.axhline(0.7, color=colors[5], linestyle='--', alpha=0.5, label='Threshold')
plt.tight_layout()
plt.show()

# 2. Best Practices for Security and Data Privacy

Production Generative AI systems must be designed with security and privacy as first-class concerns. In this section we cover:

1. Authentication & authorization
2. PII detection and redaction
3. Content safety
4. Audit logging and compliance

## 2a. Authentication and Authorization

**Never** hardcode API keys or credentials in notebooks or source code. Follow these best practices:

| Method | Use Case | Security Level |
|---|---|---|
| Environment Variables | Local development | Medium |
| Google Secret Manager | Production workloads | High |
| Workload Identity | GKE / Cloud Run | Highest |
| Application Default Credentials | Any GCP service | High |

In [ ]:
def get_secret_from_env(secret_name: str, default: str = '') -> str:
    """
    Retrieve a secret from environment variables.
    This is the minimum acceptable approach for local development.
    """
    value = os.environ.get(secret_name, default)
    if not value:
        logging.warning(
            'Secret %s not found in environment. '
            'Set it with: export %s=<your-value>', secret_name, secret_name
        )
    return value


def get_secret_from_secret_manager(project_id: str, secret_id: str,
                                   version: str = 'latest') -> str:
    """
    Retrieve a secret from Google Cloud Secret Manager.
    Requires the Secret Manager Secret Accessor IAM role.

    Usage (production):
        api_key = get_secret_from_secret_manager('my-project', 'gemini-api-key')
        genai.configure(api_key=api_key)
    """
    try:
        from google.cloud import secretmanager  # noqa: PLC0415
        client  = secretmanager.SecretManagerServiceClient()
        name    = f'projects/{project_id}/secrets/{secret_id}/versions/{version}'
        response = client.access_secret_version(request={'name': name})
        return response.payload.data.decode('UTF-8')
    except ImportError:
        logging.error('google-cloud-secret-manager not installed. '
                      'Run: pip install google-cloud-secret-manager')
        return ''
    except Exception as exc:
        logging.error('Failed to retrieve secret %s: %s', secret_id, exc)
        return ''


# Demonstrate: load API key safely
api_key = get_secret_from_env('GEMINI_API_KEY')
print(f'API key loaded: {"YES (" + "*" * 8 + api_key[-4:] + ")" if api_key else "NOT SET"}')
print()
print('IAM Best Practices:')
print('  - Grant only roles/aiplatform.user (not roles/editor or roles/owner)')
print('  - Use separate service accounts per workload')
print('  - Rotate API keys every 90 days')
print('  - Enable VPC Service Controls for sensitive data')

## 2b. PII Detection and Redaction

Before sending user data to any external API, detect and redact PII (Personally Identifiable Information). We implement a pipeline using Gemini itself as the PII detector.

Pipeline: `Raw Text → PII Detection → Redaction → API Call → Restore`

In [ ]:
class PIIRedactor:
    """
    Detects and redacts PII using Gemini, then optionally restores it
    for internal post-processing. Uses a deterministic hash map so the
    same PII token always maps to the same placeholder.
    """

    DETECT_PROMPT = """Identify all PII (Personally Identifiable Information) in the text below.
Return a JSON list of objects with keys: 'text' (the exact PII substring) and 'type'
(one of: NAME, EMAIL, PHONE, SSN, ADDRESS, CREDIT_CARD, IP_ADDRESS, DOB, OTHER).
Return [] if no PII is found. Return ONLY valid JSON, no markdown.

TEXT:
{text}"""

    def __init__(self, model_name: str = 'gemini-2.5-flash'):
        self.model    = genai.GenerativeModel(
            model_name,
            generation_config=GenerationConfig(temperature=0.0)
        )
        self._map: Dict[str, str] = {}   # placeholder → original
        self._counter: Dict[str, int] = Counter()

    def _placeholder(self, pii_type: str, original: str) -> str:
        """Deterministic placeholder based on type and a short hash."""
        short_hash = hashlib.sha1(original.encode()).hexdigest()[:6]
        return f'[{pii_type}_{short_hash}]'

    def detect(self, text: str) -> List[Dict[str, str]]:
        """Return a list of detected PII entities."""
        prompt   = self.DETECT_PROMPT.format(text=text)
        response = self.model.generate_content(prompt)
        raw      = response.text.strip()
        raw      = re.sub(r'^```json\s*|```\s*$', '', raw, flags=re.MULTILINE).strip()
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            logging.warning('PII detection returned invalid JSON: %s', raw)
            return []

    def redact(self, text: str) -> Dict[str, Any]:
        """Detect PII and return redacted text plus the mapping."""
        entities    = self.detect(text)
        redacted    = text
        mapping     = {}

        # Sort by length descending to avoid partial replacements
        entities_sorted = sorted(entities, key=lambda e: len(e['text']), reverse=True)

        for entity in entities_sorted:
            original    = entity['text']
            pii_type    = entity.get('type', 'PII')
            placeholder = self._placeholder(pii_type, original)
            mapping[placeholder] = original
            redacted = redacted.replace(original, placeholder)

        return {
            'original': text,
            'redacted': redacted,
            'entities': entities,
            'mapping':  mapping,
        }

    def restore(self, redacted_text: str, mapping: Dict[str, str]) -> str:
        """Replace placeholders with original PII values."""
        result = redacted_text
        for placeholder, original in mapping.items():
            result = result.replace(placeholder, original)
        return result

In [ ]:
# Test the PII redaction pipeline
sample_pii_text = (
    'Hello, my name is John Smith and my email is john.smith@example.com. '
    'I live at 123 Main Street, Springfield, IL 62701. '
    'Please call me at (555) 867-5309. My SSN is 123-45-6789.'
)

redactor = PIIRedactor()
result   = redactor.redact(sample_pii_text)

print('ORIGINAL TEXT:')
print(result['original'])
print()
print('DETECTED PII ENTITIES:')
for entity in result['entities']:
    print(f"  [{entity['type']}] {entity['text']}")
print()
print('REDACTED TEXT (safe to send to API):')
print(result['redacted'])
print()

# Simulate sending to API and getting a response
model          = genai.GenerativeModel('gemini-2.5-flash')
api_response   = model.generate_content(
    f'Summarize this customer inquiry in one sentence:\n{result["redacted"]}'
).text

print('API RESPONSE (with placeholders):')
print(api_response)
print()

restored = redactor.restore(api_response, result['mapping'])
print('RESTORED RESPONSE (with original PII):')
print(restored)

## 2c. Content Safety

Gemini provides built-in safety filters for harmful content categories. We can configure thresholds and must handle blocked responses gracefully.

In [ ]:
# Safety settings for a customer-facing production application
PRODUCTION_SAFETY_SETTINGS = {
    HarmCategory.HARM_CATEGORY_HARASSMENT:        HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH:        HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT:  HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT:  HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
}

# Safety settings for a research/internal tool
RESEARCH_SAFETY_SETTINGS = {
    HarmCategory.HARM_CATEGORY_HARASSMENT:        HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH:        HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT:  HarmBlockThreshold.BLOCK_HIGH_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT:  HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
}


def safe_generate(prompt: str, safety_settings: dict,
                  fallback_message: str = 'I cannot respond to that request.') -> str:
    """
    Generate content with safety settings and graceful fallback.
    Logs blocked responses for audit purposes.
    """
    model    = genai.GenerativeModel('gemini-2.5-flash', safety_settings=safety_settings)
    response = model.generate_content(prompt)

    # Check if any candidate was blocked
    if not response.candidates:
        logging.warning('[CONTENT_BLOCKED] prompt_hash=%s',
                        hashlib.sha256(prompt.encode()).hexdigest()[:16])
        return fallback_message

    candidate = response.candidates[0]
    if candidate.finish_reason.name == 'SAFETY':
        triggered = [
            f'{r.category.name}={r.probability.name}'
            for r in candidate.safety_ratings
            if r.blocked
        ]
        logging.warning('[CONTENT_BLOCKED] reason=SAFETY categories=%s', triggered)
        return fallback_message

    return response.text


# Test with a safe prompt
answer = safe_generate(
    'Explain the benefits of cloud computing in 3 bullet points.',
    safety_settings=PRODUCTION_SAFETY_SETTINGS
)
print('Safe prompt result:')
print(answer)


## 2d. Audit Logging and Compliance

For regulated industries (healthcare, finance, legal), every API call must be logged with:
- Timestamp and user identifier
- Request hash (for privacy, not the raw prompt)
- Response metadata (tokens used, finish reason)
- Any safety events

In [ ]:
class AuditLogger:
    """
    Wraps Gemini API calls with structured audit logging.
    In production, send logs to Cloud Logging or a SIEM.
    """

    def __init__(self, log_file: Optional[str] = None):
        self.log_file = log_file
        self.records: List[Dict[str, Any]] = []

        logging.basicConfig(
            format='%(asctime)s [%(levelname)s] %(message)s',
            level=logging.INFO
        )
        self.logger = logging.getLogger('AuditLogger')

    def _hash_content(self, text: str) -> str:
        """One-way hash of content for audit without storing PII."""
        return hashlib.sha256(text.encode()).hexdigest()[:16]

    def generate(self, prompt: str, model_name: str = 'gemini-2.5-flash',
                 user_id: str = 'anonymous',
                 session_id: str = '',
                 safety_settings: Optional[dict] = None) -> str:
        """Generate content and record a full audit trail entry."""
        start_time  = time.time()
        request_id  = hashlib.sha1(f'{time.time()}{user_id}'.encode()).hexdigest()[:12]

        model    = genai.GenerativeModel(model_name,
                                         safety_settings=safety_settings or {})
        response = model.generate_content(prompt)
        latency  = time.time() - start_time

        finish_reason = 'UNKNOWN'
        output_text   = ''
        if response.candidates:
            finish_reason = response.candidates[0].finish_reason.name
            if finish_reason == 'STOP':
                output_text = response.text

        record = {
            'request_id':    request_id,
            'timestamp':     time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            'user_id':       user_id,
            'session_id':    session_id,
            'model':         model_name,
            'prompt_hash':   self._hash_content(prompt),
            'prompt_length': len(prompt),
            'finish_reason': finish_reason,
            'output_length': len(output_text),
            'latency_s':     round(latency, 3),
            'blocked':       finish_reason == 'SAFETY',
        }

        self.records.append(record)
        self.logger.info('[AUDIT] %s', json.dumps(record))

        if self.log_file:
            with open(self.log_file, 'a') as fh:
                fh.write(json.dumps(record) + '\n')

        return output_text

    def get_summary(self) -> pd.DataFrame:
        """Return a summary DataFrame of all logged requests."""
        return pd.DataFrame(self.records)

In [ ]:
# Demonstrate audit logging
audit_logger = AuditLogger()

prompts_and_users = [
    ('What is machine learning?', 'user_001', 'session_abc'),
    ('Summarize the benefits of Python.', 'user_002', 'session_def'),
    ('Explain Vertex AI in one paragraph.', 'user_001', 'session_abc'),
]

for prompt, user_id, session_id in prompts_and_users:
    _ = audit_logger.generate(prompt, user_id=user_id, session_id=session_id)

summary = audit_logger.get_summary()
print('\nAudit Log Summary:')
print(summary[['request_id', 'timestamp', 'user_id', 'finish_reason',
               'prompt_length', 'output_length', 'latency_s']].to_string(index=False))

## 2e. Production Readiness Checklist

Before deploying a Generative AI application to production, verify each item below.

In [ ]:
production_checklist = {
    'Security & Auth': [
        ('No hardcoded API keys or secrets in code',           True),
        ('Secrets stored in Secret Manager',                   True),
        ('Service accounts use least-privilege IAM roles',     True),
        ('VPC Service Controls configured',                    False),
        ('API keys rotated every 90 days',                     True),
    ],
    'Data Privacy': [
        ('PII detection and redaction before API calls',       True),
        ('Data residency requirements verified',               True),
        ('User consent captured for AI processing',            True),
        ('Data retention policy defined and enforced',         False),
    ],
    'Safety & Quality': [
        ('Safety settings configured for use case',            True),
        ('Fallback responses for blocked content',             True),
        ('Output validation and post-processing',              True),
        ('Evaluation / regression test suite',                 False),
        ('Human review loop for high-risk outputs',            False),
    ],
    'Reliability': [
        ('Exponential backoff and retry logic',                True),
        ('Rate limit handling',                                True),
        ('Circuit breaker pattern implemented',                False),
        ('Graceful degradation / fallback model',              True),
    ],
    'Observability': [
        ('Structured audit logging enabled',                   True),
        ('Latency and error rate dashboards',                  True),
        ('Alerts on error rate or latency spikes',             False),
        ('Cost monitoring and budget alerts',                  True),
    ],
}

total = done = 0
for category, items in production_checklist.items():
    print(f'\n--- {category} ---')
    for item, status in items:
        mark  = '[x]' if status else '[ ]'
        print(f'  {mark} {item}')
        total += 1
        done  += int(status)

print(f'\nCompletion: {done}/{total} ({100*done//total}%)')

In [ ]:
# Visualize checklist completion by category
categories = list(production_checklist.keys())
done_counts  = [sum(1 for _, s in v if s)     for v in production_checklist.values()]
total_counts = [len(v)                         for v in production_checklist.values()]
pct          = [d / t for d, t in zip(done_counts, total_counts)]

fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = [colors[2] if p >= 0.8 else colors[3] if p >= 0.5 else colors[5] for p in pct]
bars = ax.barh(categories, pct, color=bar_colors, edgecolor='white', height=0.5)
for bar, d, t in zip(bars, done_counts, total_counts):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{d}/{t}', va='center', fontsize=10)
ax.set_xlim(0, 1.2)
ax.set_xlabel('Completion Rate', fontsize=10)
ax.set_title('Production Readiness Checklist', fontsize=12)
ax.tick_params(labelsize=10)
ax.axvline(0.8, color=colors[5], linestyle='--', alpha=0.6)
ax.text(0.81, -0.6, 'Target 80%', color=colors[5], fontsize=9)
plt.tight_layout()
plt.show()

# 3. End-to-End Example: Secure Customer Service Bot

We now combine everything into a complete application:
- RAG-powered knowledge base
- PII detection and redaction
- Content safety filters
- Full audit logging
- Graceful error handling

In [ ]:
class CustomerServiceBot:
    """
    End-to-end customer service bot that combines:
    - RAG for knowledge-grounded answers
    - PII redaction for privacy
    - Content safety settings
    - Audit logging for compliance
    - Retry logic for reliability
    """

    SYSTEM_PROMPT = (
        'You are a helpful and professional customer service agent for Acme Corp. '
        'Answer questions using only the provided context. Be concise and friendly. '
        'If you do not know the answer, say so and offer to escalate to a human agent.'
    )

    def __init__(self, knowledge_docs: List[str],
                 model_name: str = 'gemini-2.5-flash',
                 log_file: Optional[str] = None):
        self.rag          = SimpleRAG(model_name=model_name)
        self.pii_redactor = PIIRedactor(model_name=model_name)
        self.audit_logger = AuditLogger(log_file=log_file)
        self.model_name   = model_name
        self.safety_settings = PRODUCTION_SAFETY_SETTINGS

        print('Initializing knowledge base...')
        self.rag.add_documents(knowledge_docs)
        print('Customer service bot ready.')

    def _build_prompt(self, redacted_query: str, context_chunks: List[Dict]) -> str:
        context_text = '\n\n'.join(
            f'[Doc {i+1}]: {c["document"]}'
            for i, c in enumerate(context_chunks)
        )
        return (
            f'{self.SYSTEM_PROMPT}\n\n'
            f'KNOWLEDGE BASE CONTEXT:\n{context_text}\n\n'
            f'CUSTOMER QUERY: {redacted_query}\n\n'
            f'YOUR RESPONSE:'
        )

    def handle(self, user_message: str,
               user_id: str = 'anonymous',
               session_id: str = '') -> Dict[str, Any]:
        """
        Handle a customer message through the full secure pipeline.
        Returns a dict with the response and metadata.
        """
        start = time.time()

        # Step 1: PII redaction
        redaction_result = self.pii_redactor.redact(user_message)
        redacted_query   = redaction_result['redacted']
        pii_count        = len(redaction_result['entities'])

        if pii_count > 0:
            print(f'  [Privacy] Redacted {pii_count} PII entity(ies) before API call.')

        # Step 2: Retrieve relevant context
        try:
            context_chunks = self.rag.retrieve(redacted_query, top_k=3)
        except ValueError:
            context_chunks = []

        # Step 3: Build prompt and generate
        prompt = self._build_prompt(redacted_query, context_chunks)
        response_text = self.audit_logger.generate(
            prompt,
            model_name=self.model_name,
            user_id=user_id,
            session_id=session_id,
            safety_settings=self.safety_settings
        )

        if not response_text:
            response_text = ('I\'m sorry, I cannot assist with that request. '
                             'Please contact our support team at support@acme.com.')

        return {
            'user_id':     user_id,
            'session_id':  session_id,
            'query':       user_message,
            'response':    response_text,
            'pii_removed': pii_count,
            'sources':     len(context_chunks),
            'latency_s':   round(time.time() - start, 3),
        }

In [ ]:
# Acme Corp knowledge base
ACME_DOCS = [
    'Acme Corp offers three support tiers: Basic (email, 48h SLA), '
    'Pro (email + chat, 8h SLA), and Enterprise (24/7 phone, 1h SLA).',

    'Refund policy: Products can be returned within 30 days of purchase for a full refund. '
    'Software licenses are non-refundable after activation.',

    'Our Laptop Pro model has a 3-year warranty. Accidental damage coverage is '
    'available as an add-on for $99/year.',

    'To reset your account password, visit account.acme.com/reset and enter your '
    'registered email address. A reset link is valid for 24 hours.',

    'Shipping times: Standard (5-7 business days, free over $50), '
    'Express (2-3 business days, $15), Overnight ($35).',

    'Our business hours are Monday-Friday 9am-6pm ET. '
    'Enterprise customers have 24/7 dedicated support.',
]

bot = CustomerServiceBot(ACME_DOCS, log_file='customer_service_audit.jsonl')

In [ ]:
# Simulate customer conversations
conversations = [
    (
        'Hi, I am Jane Doe (jane.doe@gmail.com). '
        'I bought a Laptop Pro last week. What is the warranty?',
        'user_jane', 'sess_001'
    ),
    (
        'How do I get a refund? My order number is 987654.',
        'user_bob', 'sess_002'
    ),
    (
        'What are your business hours?',
        'user_alice', 'sess_003'
    ),
]

results = []
for message, user_id, session_id in conversations:
    print('=' * 65)
    print(f'Customer ({user_id}): {message}')
    result = bot.handle(message, user_id=user_id, session_id=session_id)
    print(f'Bot Response: {result["response"]}')
    print(f'  [Meta] PII removed: {result["pii_removed"]}, '
          f'sources: {result["sources"]}, latency: {result["latency_s"]}s')
    results.append(result)
    print()

In [ ]:
# Show end-to-end architecture diagram
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14); ax.set_ylim(0, 9); ax.axis('off')
ax.set_title('Secure Customer Service Bot – Architecture', fontsize=13, fontweight='bold')

arch_colors = {
    'user':    colors[0],
    'pii':     colors[5],
    'rag':     colors[1],
    'llm':     colors[2],
    'audit':   colors[3],
    'output':  colors[6],
    'storage': colors[7],
}

def abox(ax, x, y, w, h, label, color, fontsize=10):
    rect = mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle='round,pad=0.08',
        facecolor=color, edgecolor='white', linewidth=2, alpha=0.9, zorder=3
    )
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=fontsize,
            color='white', fontweight='bold', zorder=4)

def aarrow(ax, x1, y1, x2, y2, color=None, label=''):
    if color is None:
        color = colors[7]
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8), zorder=2)
    if label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx, my + 0.18, label, ha='center', fontsize=9,
                color=color, style='italic')

# Draw nodes
abox(ax,  2,   7,   2.8, 0.9, 'Customer',              arch_colors['user'],    10)
abox(ax,  2,   5,   2.8, 0.9, 'PII Redactor',          arch_colors['pii'],     10)
abox(ax,  2,   3,   2.8, 0.9, 'RAG Retriever',         arch_colors['rag'],     10)
abox(ax,  7,   5,   2.8, 0.9, 'Gemini 2.0 Flash',      arch_colors['llm'],     10)
abox(ax,  7,   3,   2.8, 0.9, 'Audit Logger',          arch_colors['audit'],   10)
abox(ax, 12,   5,   2.8, 0.9, 'Response',              arch_colors['output'],  10)
abox(ax,  2,   1,   2.8, 0.9, 'Vector DB',             arch_colors['storage'], 10)
abox(ax,  7,   1,   2.8, 0.9, 'Cloud Logging',         arch_colors['storage'], 10)
abox(ax, 12,   3,   2.8, 0.9, 'Secret Manager',        arch_colors['pii'],     10)

# Draw arrows
aarrow(ax, 2, 6.55, 2, 5.45, label='raw query')
aarrow(ax, 2, 4.55, 2, 3.45, label='redacted query')
aarrow(ax, 2, 2.55, 2, 1.45, label='embed query')
aarrow(ax, 2, 1.45, 2, 2.55, arch_colors['rag'], 'top-k chunks')
aarrow(ax, 3.4, 5,  5.6, 5,  label='prompt + context')
aarrow(ax, 7, 4.55, 7, 3.45, label='log request')
aarrow(ax, 7, 2.55, 7, 1.45, label='write log')
aarrow(ax, 8.4, 5, 10.6, 5,  arch_colors['llm'], 'generated text')
aarrow(ax, 12, 4.55, 12, 3.45, arch_colors['pii'], 'fetch API key')
aarrow(ax, 3.4, 3,  5.6, 4.7, arch_colors['rag'], 'retrieved docs')

plt.tight_layout()
plt.savefig('e2e_architecture.png', dpi=120, bbox_inches='tight')
plt.show()

# Summary

In this segment we built a complete, production-ready Generative AI solution using the Gemini API and Vertex AI. Key takeaways:

| Topic | Key Points |
|---|---|
| **RAG** | Grounds LLM answers in real knowledge; uses vector similarity retrieval |
| **Agents** | Function calling lets Gemini orchestrate tools dynamically |
| **Pipelines** | Chained LLM calls for complex, multi-stage tasks |
| **Evaluation** | LLM-as-judge provides scalable, automated quality assessment |
| **Security** | Never hardcode secrets; use Secret Manager + least-privilege IAM |
| **Privacy** | Detect and redact PII before every external API call |
| **Safety** | Configure safety thresholds per use case; always handle blocked responses |
| **Audit** | Log every request with hashed content for compliance without storing PII |

**Next Steps:**
- Replace the in-memory vector store with Vertex AI Vector Search for scale
- Deploy the bot as a Cloud Run service with Workload Identity
- Add a streaming response interface for lower perceived latency
- Set up Cloud Monitoring dashboards and alerting policies

<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>